In [ ]:
# =========================================================
# IMPORTS
# =========================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix

from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# =========================================================
# LOAD DATA
# =========================================================
df = pd.read_csv("lte_7attacks_augmented.csv")

In [ ]:
# =========================================================
# FEATURE ENGINEERING
# =========================================================
df['LostPackets'] = df['TxPackets'] - df['RxPackets']
df['LossRate'] = df['LostPackets'] / (df['TxPackets'] + 1)
df['Traffic'] = df['TxPackets'] + df['RxPackets']
df['Diff'] = df['TxPackets'] - df['RxPackets']

features = [col for col in df.columns if col not in ['Label', 'AttackType']]
X = df[features].values

le = LabelEncoder()
y = le.fit_transform(df['Label'])

class_names = le.classes_
num_classes = len(class_names)

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# =========================================================
# LSTM DATA
# =========================================================
def create_sequences(X, y, window=15):
    X_seq, y_seq = [], []
    for i in range(len(X) - window):
        X_seq.append(X[i:i+window])
        y_seq.append(y[i+window-1])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y)

X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_seq, y_seq, test_size=0.2, stratify=y_seq
)

In [ ]:
# =========================================================
# LSTM MODEL
# =========================================================
lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_seq.shape[1], X_seq.shape[2])),
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining LSTM...")
history = lstm_model.fit(X_train_lstm, y_train_lstm, epochs=30, batch_size=32)

In [ ]:
# =========================================================
# TABTRANSFORMER
# =========================================================
X_train_tab, X_test_tab, y_train_tab, y_test_tab = train_test_split(
    X, y, test_size=0.2, stratify=y
)

X_train_tab = torch.tensor(X_train_tab, dtype=torch.float32)
X_test_tab  = torch.tensor(X_test_tab,  dtype=torch.float32)
y_train_tab = torch.tensor(y_train_tab, dtype=torch.long)
y_test_tab  = torch.tensor(y_test_tab,  dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train_tab, y_train_tab),
    batch_size=64, shuffle=True
)

class TabTransformer(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.embed = nn.Linear(1, 64)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64, nhead=8, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=3)
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Sequential(
            nn.Linear(64 * input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)
        x = self.embed(x)
        x = self.transformer(x)
        x = self.dropout(x)
        x = x.flatten(start_dim=1)
        return self.fc(x)

tab_model = TabTransformer(X.shape[1], num_classes)
criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(tab_model.parameters(), lr=0.0003)

print("\nTraining TabTransformer...")
epochs    = 30
tab_losses = []
tab_acc    = []

for epoch in range(epochs):
    tab_model.train()
    total_loss = 0
    correct = 0
    total   = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        out  = tab_model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds   = torch.argmax(out, dim=1)
        correct += (preds == yb).sum().item()
        total   += yb.size(0)

    acc = correct / total
    tab_losses.append(total_loss / total)
    tab_acc.append(acc)
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Acc: {acc:.4f}")

In [ ]:
# =========================================================
# BEFORE ATTACK RESULTS
# =========================================================
y_pred_lstm = np.argmax(lstm_model.predict(X_test_lstm), axis=1)

tab_model.eval()
with torch.no_grad():
    preds_tab = torch.argmax(tab_model(X_test_tab), dim=1)

print("\n BEFORE ATTACK (%)")
print("LSTM: {:.2f}%".format(accuracy_score(y_test_lstm, y_pred_lstm)*100))
print("TAB : {:.2f}%".format(accuracy_score(y_test_tab.numpy(), preds_tab.numpy())*100))

In [ ]:
# =========================================================
# FGSM ATTACK
# =========================================================
def fgsm_lstm(model, X, y, eps=0.05):
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    y_t = tf.convert_to_tensor(y)
    with tf.GradientTape() as tape:
        tape.watch(X_t)
        pred = model(X_t)
        loss = tf.keras.losses.sparse_categorical_crossentropy(y_t, pred)
    grad = tape.gradient(loss, X_t)
    return (X_t + eps * tf.sign(grad)).numpy()

def fgsm_tab(model, X, y, eps=0.05):
    X_adv = X.clone().detach().requires_grad_(True)
    out   = model(X_adv)
    loss  = nn.CrossEntropyLoss()(out, y)
    loss.backward()
    return (X_adv + eps * X_adv.grad.sign()).detach()

# Apply attack
X_adv_lstm = fgsm_lstm(lstm_model, X_test_lstm, y_test_lstm)
y_adv_lstm = np.argmax(lstm_model.predict(X_adv_lstm), axis=1)

X_adv_tab = fgsm_tab(tab_model, X_test_tab, y_test_tab)
with torch.no_grad():
    preds_adv_tab = torch.argmax(tab_model(X_adv_tab), dim=1)

In [ ]:
# =========================================================
# AFTER ATTACK RESULTS
# =========================================================
print("\n AFTER ATTACK (%)")
print("LSTM: {:.2f}%".format(accuracy_score(y_test_lstm, y_adv_lstm)*100))
print("TAB : {:.2f}%".format(accuracy_score(y_test_tab.numpy(), preds_adv_tab.numpy())*100))

In [ ]:
# =========================================================
# CONFUSION MATRIX
# =========================================================
def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title(title)
    plt.show()

plot_cm(y_test_lstm,       y_pred_lstm,           "LSTM Before Attack")
plot_cm(y_test_lstm,       y_adv_lstm,            "LSTM After Attack")
plot_cm(y_test_tab.numpy(), preds_tab.numpy(),     "TabTransformer Before Attack")
plot_cm(y_test_tab.numpy(), preds_adv_tab.numpy(), "TabTransformer After Attack")

In [ ]:
# =========================================================
# GRAPHS
# =========================================================
# Accuracy
plt.figure()
plt.plot(history.history['accuracy'], label='LSTM Accuracy')
plt.plot(tab_acc, label='TabTransformer Accuracy')
plt.legend()
plt.title('Accuracy Comparison')
plt.show()

# Normalized Loss
plt.figure()
lstm_loss = np.array(history.history['loss']) / max(history.history['loss'])
tab_loss  = np.array(tab_losses) / max(tab_losses)
plt.plot(lstm_loss, label='LSTM Loss')
plt.plot(tab_loss,  label='TabTransformer Loss')
plt.legend()
plt.title('Loss Comparison (Normalized)')
plt.show()